In [ ]:
# 1 Libraries
!pip install -q kagglehub opencv-python torch torchvision scikit-learn matplotlib

import os, cv2, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F  ####
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import kagglehub
from tqdm import tqdm
import random  ####
from torchvision.models import resnet18, ResNet18_Weights ####
from tensorflow.keras.applications import MobileNetV2
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
# 2 DATASET
dataset_dir = kagglehub.dataset_download("ruizgara/socofing")
real_path = os.path.join(dataset_dir, "SOCOFing", "Real")
assert os.path.isdir(real_path), "Real folder not found!"
#Collect BMP files
file_list = []
for root, _, files in os.walk(real_path):
    for f in files:
        if f.lower().endswith(".bmp"):
            file_list.append(os.path.join(root, f))
print(f"Total BMP images found: {len(file_list)}")

Using Colab cache for faster access to the 'socofing' dataset.
Total BMP images found: 6000


In [ ]:
# 3 Preprocess
import cv2
import matplotlib.pyplot as plt
import numpy as np # Needed for np.expand_dims

# IMPORTANT: Function signature changed to accept path, size changed to 224x224
def preprocess(img_path, size=(224, 224)):
    # Load the image in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # 1. Resize with anti-aliasing
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

    # 2. Light smoothing
    img = cv2.GaussianBlur(img, (1, 1), 0)

    # 3. Subtle contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4))
    img = clahe.apply(img)

    # 4. Normalize pixel values
    img = img.astype(np.float32) / 255.0

    # 5. Add a channel dimension: (H, W) -> (H, W, 1)
    img = np.expand_dims(img, axis=-1) # represent height, width and channel

    return img

sample_path = file_list[0] # first image in the list
processed = preprocess(sample_path)
print(f"Processed image shape: {processed.shape}")

Processed image shape: (224, 224, 1)


In [ ]:
# 4 PyTorch Dataset & DataLoader
class FingerprintDataset(Dataset):
    def __init__(self, file_list, preprocess, size=(224,224)):
        self.file_list = file_list
        self.preprocess = preprocess
        self.size = size

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        img = self.preprocess(img_path, size=self.size)
        img = torch.tensor(img.transpose(2,0,1))  # HWC → CHW
        return img

# Create dataset & dataloader
batch_size = 32
dataset = FingerprintDataset(file_list, preprocess)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)


In [ ]:
#Feature Extraction (MobileNetv2)

def extract_features(dataloader, model):
    """
    Extracts features from all images in the dataloader using the provided model.
    """
    model.eval() # Set model to evaluation mode
    all_features = []

    # Determine the device (GPU or CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    with torch.no_grad(): # Disable gradient calculation for inference
        for inputs in tqdm(dataloader, desc="Extracting Features"):

            # The input is (N, 1, H, W). Triplicate the single channel (1)
            # into three channels (3) for the pre-trained model.
            inputs = inputs.float()
            inputs_3_channel = inputs.repeat(1, 3, 1, 1)

            # Move data to the same device as the model
            inputs_3_channel = inputs_3_channel.to(device)

            # Get features
            features = model(inputs_3_channel)

            # Flatten the features
            # (e.g., from (32, 1280, 1, 1) to (32, 1280))
            features = features.squeeze().cpu().numpy()
            all_features.append(features)

    # Concatenate all batches into a single numpy array
    all_features = np.concatenate(all_features, axis=0)
    return all_features

model = models.mobilenet_v2(pretrained=True)

# Remove the classifier layer (the head) to get the features.
feature_extractor = model.features

# --- Run Extraction ---
print("Starting feature extraction...")

# Assuming 'dataloader' was defined in the previous cell
fingerprint_features = extract_features(dataloader, feature_extractor)

print(f"\nFeatures extracted successfully!")
# Output shape should be (6000, 1280) if you have 6000 images.
print(f"Final feature array shape: {fingerprint_features.shape}")

np.save("mobilenet_features.npy", fingerprint_features)
print("Features saved to 'mobilenet_features.npy'.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 136MB/s]


Starting feature extraction...


Extracting Features: 100%|██████████| 188/188 [01:02<00:00,  3.01it/s]



Features extracted successfully!
Final feature array shape: (6000, 1280, 7, 7)
Features saved to 'mobilenet_features.npy'.


In [ ]:
# =========================================================
# VAE Feature Extraction (Latent Space Features)
# =========================================================

import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),    # 112
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),   # 56
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1),  # 28
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), # 14
            nn.ReLU(),
            nn.Conv2d(256, 512, 4, 2, 1), # 7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 1024),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(1024, latent_dim)
        self.fc_logvar = nn.Linear(1024, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (512, 7, 7)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

# ------------------- Train VAE (quick training) -------------------
def vae_loss(recon_x, x, mu, logvar):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VAE(latent_dim=128).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)

print("Training VAE on SOCOFing (this may take 5-15 minutes)...")
vae.train()
for epoch in range(20):  # 20 epochs is enough for decent latent features
    total_loss = 0
    for batch in tqdm(dataloader, desc=f"VAE Epoch {epoch+1}/20"):
        batch = batch.to(device).float()

        recon, mu, logvar, z = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar)

        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1} - Loss: {total_loss / len(dataset):.2f}")

# ------------------- Extract VAE latent features -------------------
print("Extracting VAE latent features...")
vae.eval()
vae_features = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="VAE Feature Extraction"):
        batch = batch.to(device).float()
        mu, logvar = vae.encode(batch)
        z = vae.reparameterize(mu, logvar)  # or just use mu
        vae_features.append(z.cpu().numpy())

vae_features = np.concatenate(vae_features, axis=0)
print(f"VAE features shape: {vae_features.shape}")  # (6000, 128)

np.save("vae_features.npy", vae_features)
print("VAE features saved to 'vae_features.npy'")

Training VAE on SOCOFing (this may take 5-15 minutes)...


VAE Epoch 1/20: 100%|██████████| 188/188 [00:24<00:00,  7.75it/s]


Epoch 1 - Loss: 26466.71


VAE Epoch 2/20: 100%|██████████| 188/188 [00:23<00:00,  8.11it/s]


Epoch 2 - Loss: 25731.13


...

Epoch 19 - Loss: 25604.16


VAE Epoch 20/20: 100%|██████████| 188/188 [00:23<00:00,  8.10it/s]


Epoch 20 - Loss: 25606.61
Extracting VAE latent features...


VAE Feature Extraction: 100%|██████████| 188/188 [00:11<00:00, 17.07it/s]

VAE features shape: (6000, 128)
VAE features saved to 'vae_features.npy'


In [ ]:
# =========================================================
# SIMCLR – Feature Extraction
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import resnet18, ResNet18_Weights
from tqdm import tqdm
import numpy as np
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------- SimCLR Model -------------------
class SimCLR(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        self.encoder = backbone
        self.projector = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, feature_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)

# ------------------- Augmentations (GRAYSCALE → 3-channel) -------------------
class SimCLRAugment:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size=size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=23)], p=0.5),
            transforms.Lambda(lambda x: x.convert("RGB")),           # ← FORCE 3 channels
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],        # ← ImageNet stats
                                 std=[0.229, 0.224, 0.225]),
        ])
    def __call__(self, pil_img):
        return self.transform(pil_img), self.transform(pil_img)

# ------------------- Dataset -------------------
class SimCLRFingerprintDataset(Dataset):
    def __init__(self, file_list, preprocess_fn):
        self.file_list = file_list
        self.preprocess = preprocess_fn
        self.augment = SimCLRAugment()

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        img = self.preprocess(path)                     # (224,224,1) float32 [0,1]
        img = (img * 255).astype(np.uint8).squeeze(-1)  # (224,224) uint8
        pil_img = Image.fromarray(img)                  # Grayscale PIL
        view1, view2 = self.augment(pil_img)
        return view1, view2

# ------------------- DataLoader (num_workers=0 is mandatory) -------------------
simclr_dataset = SimCLRFingerprintDataset(file_list, preprocess)
simclr_loader = DataLoader(
    simclr_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,       # ← NO multiprocessing → no CUDA worker crash
    pin_memory=False,
    drop_last=True
)

# ------------------- Stable NT-Xent Loss -------------------
def nt_xent_loss(z1, z2, temperature=0.5):
    z = torch.cat([z1, z2], dim=0)                          # (2N, D)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=-1) / temperature
    sim = sim - torch.eye(z.shape[0], device=z.device) * 1e12

    labels = torch.cat([torch.arange(z1.shape[0]) + z1.shape[0], torch.arange(z1.shape[0])])
    labels = labels.to(z.device)

    loss = F.cross_entropy(sim, labels)
    return loss

# ------------------- Training -------------------
model = SimCLR(feature_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-6)

print("Starting SimCLR training (15 epochs)...")
model.train()
for epoch in range(1, 16):
    epoch_loss = 0.0
    for view1, view2 in tqdm(simclr_loader, desc=f"Epoch {epoch}/15"):
        view1, view2 = view1.to(device), view2.to(device)

        z1 = model(view1)
        z2 = model(view2)
        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch:02d} - Loss: {epoch_loss/len(simclr_loader):.4f}")

print("SimCLR training finished!\n")

# ------------------- Feature Extraction (512-dim) -------------------
print("Extracting SimCLR features (512-dim from encoder)...")

class ExtractDataset(Dataset):
    def __len__(self): return len(file_list)
    def __getitem__(self, idx):
        img = preprocess(file_list[idx])                      # (224,224,1)
        img = torch.from_numpy(img.transpose(2,0,1)).float()  # (1,224,224)
        return img.repeat(3,1,1)                              # → (3,224,224)

extract_loader = DataLoader(ExtractDataset(), batch_size=128, shuffle=False, num_workers=0)

model.eval()
features = []
with torch.no_grad():
    for batch in tqdm(extract_loader, desc="Extracting"):
        h = model.encoder(batch.to(device))
        features.append(h.cpu().numpy())

simclr_features = np.concatenate(features)
print(f"SimCLR features shape: {simclr_features.shape}")   # (6000, 512)

# Save
np.save("simclr_features.npy", simclr_features)
print("simclr_features.npy saved!")
torch.save(model.state_dict(), "simclr_fingerprint_model.pth")
print("Model saved as simclr_fingerprint_model.pth")

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 31.0MB/s]


Starting SimCLR training (15 epochs)...


Epoch 1/15: 100%|██████████| 46/46 [06:25<00:00,  8.38s/it]


Epoch 01 - Loss: 4.4040


Epoch 2/15: 100%|██████████| 46/46 [06:14<00:00,  8.13s/it]


Epoch 02 - Loss: 4.0740


Epoch 3/15:  91%|█████████▏| 42/46 [05:51<00:34,  8.53s/it]

In [ ]:
# =========================================================
# 1. MobileNetV2 – Dimension Reduction & Clustering
# =========================================================

# --- 1. Load and Flatten Features ---
print("Loading and flattening features...")
fingerprint_features = np.load("mobilenet_features.npy")
# Flatten from (6000, 1280, 7, 7) to (6000, 62720)
flattened_features = fingerprint_features.reshape(fingerprint_features.shape[0], -1)
print(f"Features flattened to shape: {flattened_features.shape}")


print("Applying PCA to reduce dimensions to 100...")
n_components = 100
pca = PCA(n_components=n_components, random_state=42)
pca_features = pca.fit_transform(flattened_features)
print(f"PCA reduced features shape: {pca_features.shape}")


# --- 3. Clustering on PCA Features ---
# K-Means runs much faster on 100 features than on 62,720 features.
print("Applying K-Means clustering on PCA features...")
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(pca_features)
print("Clustering complete.")


# --- 4. t-SNE Visualization on PCA Features ---
# t-SNE is applied to the 100 PCA features.
print("Applying t-SNE (Projection from 100 dimensions to 2)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_results = tsne.fit_transform(pca_features) # Using pca_features here!
print("Dimensionality reduction complete.")

# Save the final results for plotting
np.save("tsne_results.npy", tsne_results)
np.save("cluster_labels.npy", cluster_labels)

In [ ]:
# =========================================================
# 1. VAE – Dimension Reduction & Clustering
# =========================================================

import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Load VAE latent features (already low-dim: 6000×128)
vae_features = np.load("vae_features.npy")
print(f"VAE raw features shape : {vae_features.shape}")   # (6000, 128)

# VAE latent space is already very compact → we can skip heavy PCA
# But we still apply a light PCA (50 components) for perfect fairness with the others
print("Applying light PCA (50 components) on VAE latent space...")
pca_vae = PCA(n_components=50, random_state=42)
vae_pca = pca_vae.fit_transform(vae_features)
print(f"VAE PCA-reduced shape   : {vae_pca.shape}")        # (6000, 50)

# K-Means (600 true fingers → k=600, or use 600 for fair comparison)
k = 600
print(f"Running K-Means (k={k}) on VAE PCA features...")
kmeans_vae = KMeans(n_clusters=k, random_state=42, n_init=10)
vae_labels = kmeans_vae.fit_predict(vae_pca)

# t-SNE on the 50-dim PCA features
print("Running t-SNE on VAE PCA features...")
tsne_vae = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
tsne_vae_2d = tsne_vae.fit_transform(vae_pca)

# Save
np.save("vae_pca.npy", vae_pca)
np.save("vae_cluster_labels.npy", vae_labels)
np.save("vae_tsne_2d.npy", tsne_vae_2d)
print("VAE pipeline finished!\n")

In [ ]:
# =========================================================
# 3. SimCLR – Dimension Reduction & Clustering
# =========================================================
import numpy as np
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.cluster import KMeans # Import KMeans

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class QuickDataset(Dataset):
    def __len__(self): return len(file_list)
    def __getitem__(self, idx):
        img = preprocess(file_list[idx])                  # → (224,224,1) numpy
        img = torch.tensor(img.transpose(2,0,1))          # → (1,224,224) tensor
        return img

loader = DataLoader(QuickDataset(), batch_size=128, shuffle=False, num_workers=2)

# Extract features using the trained backbone
print("Extracting SimCLR features (512-dim) from trained model...")
simclr_model.eval()
features = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting"):
        batch = batch.to(device).float()
        batch = batch.repeat(1, 3, 1, 1)           # (B,1,224,224) → (B,3,224,224)
        h = simclr_model.backbone(batch)
        h = F.adaptive_avg_pool2d(h, 1).flatten(1)  # → (B,512)
        features.append(h.cpu().numpy())

features = np.concatenate(features, axis=0)
print(f"Done! Shape: {features.shape}")

# SAVE IT
np.save("simclr_features.npy", features)
print("simclr_features.npy CREATED SUCCESSFULLY!")

# Perform K-Means clustering with k=600 and save labels
k = 600 # Use k=600 for consistency with other models
kmeans_simclr = KMeans(n_clusters=k, random_state=42, n_init=10)
simclr_cluster_labels = kmeans_simclr.fit_predict(features)
np.save("simclr_cluster_labels.npy", simclr_cluster_labels)
print("simclr_cluster_labels.npy CREATED SUCCESSFULLY!")

In [ ]:
# =========================================================
# 1. MobileNetV2 – Dimension Reduction & Clustering  (K=3)
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
cluster_labels_k3 = kmeans.fit_predict(pca_features)
print("Clustering complete with K=3.")

pca_age = PCA(n_components=1, random_state=42)
age_proxy = pca_age.fit_transform(flattened_features).squeeze()

age_bins = [age_proxy.min()-1, np.percentile(age_proxy, 33.3), np.percentile(age_proxy, 66.6), age_proxy.max()+1]

# Assign semantic labels
age_labels_k3 = ["Child/Youth", "Adult", "Aged"]
age_group_k3 = pd.cut(age_proxy, bins=age_bins, labels=age_labels_k3, include_lowest=True)

# --- 6. Compute distribution of inferred age groups per cluster ---
df_k3 = pd.DataFrame({
    "cluster": cluster_labels_k3,
    "age_group": age_group_k3
})

cluster_age_distribution_k3 = df_k3.groupby(["cluster", "age_group"]).size().unstack(fill_value=0)
print("\nFinal Inferred Age Distribution per OPTIMAL K=3 Cluster:")
print(cluster_age_distribution_k3)
# Visualization
plt.figure(figsize=(8, 5))
custom_colors = ['red', 'yellow', 'purple']
cluster_age_distribution_k3.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=custom_colors,
    ax=plt.gca()
)

plt.title("Final Unsupervised Age Inference: Optimal K=3 Clusters")
plt.xlabel("K-Means Cluster=3")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.legend(title="Inferred Age Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("unsupervised_age_inference_k3_final.png")
print("Final K=3 age inference visualization saved as unsupervised_age_inference_k3_final.png")

In [ ]:
# =========================================================
# 2. VAE – Dimension Reduction & Clustering  (K=3)
# =========================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load VAE features and use them directly (already compact)
vae_features = np.load("vae_features.npy")                     # (6000, 128)

# 1. K=3 clustering on VAE latent space
kmeans_vae = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_vae_k3 = kmeans_vae.fit_predict(vae_features)

# 2. 1D PCA → age proxy
pca_age = PCA(n_components=1, random_state=42)
age_proxy = pca_age.fit_transform(vae_features).squeeze()

# 3. Split into 3 age groups (33rd and 66th percentiles)
age_bins = [age_proxy.min()-1, np.percentile(age_proxy, 33.3),
            np.percentile(age_proxy, 66.6), age_proxy.max()+1]
age_labels = ["Child/Youth", "Adult", "Aged"]
age_group = pd.cut(age_proxy, bins=age_bins, labels=age_labels, include_lowest=True)

# 4. Distribution
df = pd.DataFrame({"cluster": cluster_labels_vae_k3, "age_group": age_group})
dist = df.groupby(["cluster", "age_group"]).size().unstack(fill_value=0)

print("VAE – K=3 Age Inference Distribution:")
print(dist)

# 5. Plot
plt.figure(figsize=(10, 6))
dist.plot(kind="bar", stacked=True, color=['red', 'yellow', 'purple'], ax=plt.gca())
plt.title("VAE → Unsupervised Age Inference (K=3)", fontsize=14, fontweight='bold')
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.legend(title="Inferred Age Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("vae_unsupervised_age_inference_k3.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: vae_unsupervised_age_inference_k3.png")

In [ ]:
# =========================================================
# 3. SimCLR – Dimension Reduction & Clustering  (K=3)
# =========================================================


import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load SimCLR features
simclr_features = np.load("simclr_features.npy")               # (6000, 512)

# 1. K=3 clustering
kmeans_simclr = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels_simclr_k3 = kmeans_simclr.fit_predict(simclr_features)

# 2. 1D PCA → age proxy
pca_age = PCA(n_components=1, random_state=42)
age_proxy = pca_age.fit_transform(simclr_features).squeeze()

# 3. Same 3 age bins
age_bins = [age_proxy.min()-1, np.percentile(age_proxy, 33.3),
            np.percentile(age_proxy, 66.6), age_proxy.max()+1]
age_labels = ["Child/Youth", "Adult", "Aged"]
age_group = pd.cut(age_proxy, bins=age_bins, labels=age_labels, include_lowest=True)

# 4. Distribution
df = pd.DataFrame({"cluster": cluster_labels_simclr_k3, "age_group": age_group})
dist = df.groupby(["cluster", "age_group"]).size().unstack(fill_value=0)

print("SimCLR – K=3 Age Inference Distribution:")
print(dist)

# 5. Plot
plt.figure(figsize=(10, 6))
dist.plot(kind="bar", stacked=True, color=['red', 'yellow', 'purple'], ax=plt.gca())
plt.title("SimCLR → Unsupervised Age Inference (K=3)", fontsize=14, fontweight='bold')
plt.xlabel("Cluster")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.legend(title="Inferred Age Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("simclr_unsupervised_age_inference_k3.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: simclr_unsupervised_age_inference_k3.png")

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ------------------------------------------------------------------
# Load features (change paths if needed)
# ------------------------------------------------------------------
mobilenet_feats = np.load("mobilenet_features.npy")      # (6000, 1280, 7, 7)
vae_feats       = np.load("vae_features.npy")            # (6000, 128)
simclr_feats    = np.load("simclr_features.npy")         # whatever you have

# MobileNetV2 → Global Average Pooling
if mobilenet_feats.ndim == 4:
    mobilenet_feats = mobilenet_feats.mean(axis=(2, 3))  # → (6000, 1280)

print(f"MobileNetV2 shape : {mobilenet_feats.shape}")
print(f"VAE shape         : {vae_feats.shape}")
print(f"SimCLR shape      : {simclr_feats.shape}")

feature_sets = {
    "MobileNetV2": mobilenet_feats,
    "VAE"        : vae_feats,
    "SimCLR"     : simclr_feats
}

# ------------------------------------------------------------------
# Comparison function (now bullet-proof)
# ------------------------------------------------------------------
def compare_all_features(feature_dict,
                         k_range=range(2, 15),
                         pca_components=50,
                         tsne_perplexity=30,
                         min_samples_for_tsne=50):

    results = {}
    silhouette_curves = {}

    for name, feats in feature_dict.items():
        print(f"\n=== Processing {name} ===")
        n_samples, n_features = feats.shape
        print(f"Samples: {n_samples} | Features: {n_features}")

        # --------------------------------------------------------------
        # 1. PCA reduction (but never ask for more components than exist)
        # --------------------------------------------------------------
        max_pca = min(pca_components, n_features, n_samples-1)
        if max_pca < 2:
            print(f"  → Not enough variance for PCA → using raw features")
            feats_reduced = feats
        else:
            pca = PCA(n_components=max_pca, random_state=42)
            feats_reduced = pca.fit_transform(feats)
            print(f"  → PCA → {feats_reduced.shape}")

        # --------------------------------------------------------------
        # 2. Silhouette analysis over K
        # --------------------------------------------------------------
        sil_scores = []
        for k in tqdm(k_range, desc="KMeans silhouette"):
            if k >= n_samples:           # cannot have more clusters than points
                sil_scores.append(-1)
                continue
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            labels = kmeans.fit_predict(feats_reduced)
            score = silhouette_score(feats_reduced, labels)
            sil_scores.append(score)

        best_k = k_range[np.argmax(sil_scores)]
        best_score = max(sil_scores)

        results[name] = {"best_k": best_k, "best_silhouette": best_score}
        silhouette_curves[name] = sil_scores

        print(f"  → Best K = {best_k} | Silhouette = {best_score:.4f}")

        # --------------------------------------------------------------
        # 3. t-SNE only if we have enough samples & dimensions
        # --------------------------------------------------------------
        if n_samples >= min_samples_for_tsne and max_pca >= 2:
            tsne = TSNE(n_components=2,
                        perplexity=min(tsne_perplexity, n_samples//3),
                        random_state=42,
                        n_iter=1000)
            tsne_emb = tsne.fit_transform(feats_reduced)

            kmeans_vis = KMeans(n_clusters=best_k, random_state=42, n_init=10)
            labels_vis = kmeans_vis.fit_predict(feats_reduced)

            plt.figure(figsize=(9, 7))
            scatter = plt.scatter(tsne_emb[:, 0], tsne_emb[:, 1],
                                  c=labels_vis, cmap='tab10', s=15, alpha=0.8)
            plt.title(f"t-SNE – {name} (K={best_k}, sil={best_score:.3f})")
            plt.colorbar(scatter)
            plt.tight_layout()
            fname = f"tsne_{name.lower().replace(' ', '_')}.png"
            plt.savefig(fname, dpi=200)
            plt.show()
            print(f"  → t-SNE saved as {fname}")
        else:
            print(f"  → Skipping t-SNE (only {n_samples} samples)")

    # ------------------------------------------------------------------
    # Final summary table + silhouette plot
    # ------------------------------------------------------------------
    print("\n" + "="*60)
    print("FINAL MODEL COMPARISON")
    print("="*60)
    print(f"{'Model':<15} {'Best K':<8} {'Silhouette':<12} {'Samples'}")
    print("-"*60)
    for name in feature_dict:
        r = results[name]
        n = feature_dict[name].shape[0]
        print(f"{name:<15} {r['best_k']:<8} {r['best_silhouette']:<12.4f} {n}")
    print("-"*60)

    # Silhouette vs K curve
    plt.figure(figsize=(10, 6))
    for name, scores in silhouette_curves.items():
        plt.plot(list(k_range), scores, marker='o', label=name)
    plt.xlabel("Number of clusters K")
    plt.ylabel("Silhouette score")
    plt.title("Silhouette score for different K")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("silhouette_vs_k_comparison.png", dpi=200)
    plt.show()

# ------------------------------------------------------------------
# Run it!
# ------------------------------------------------------------------
compare_all_features(feature_sets)

In [ ]:
# ←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←
# FIX: SIMCLR ON FULL 6000 FINGERPRINTS
# ←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import cv2

print(f"Using the FULL dataset → {len(file_list)} images")

# Strong SimCLR augmentations (grayscale version)
def get_simclr_augs():
    return transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.8, contrast=0.8)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.GaussianBlur(kernel_size=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485], std=[0.229])  # will be repeated to 3 channels later
    ])

class SimCLRFingerprintDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, axis=-1)  # (224,224,1)

        if self.transform:
            aug1 = self.transform(img)
            aug2 = self.transform(img)
        else:
            t = transforms.ToTensor()
            aug1 = aug2 = t(img)

        return aug1, aug2

# Load official pretrained SimCLR-v2 ResNet50 (best publicly available)
print("Loading pretrained SimCLR ResNet50...")
simclr_model = torch.hub.load('facebookresearch/semi-supervised-ImageNet1K-models',
                              'resnet50_ssl', pretrained=True)
simclr_model.eval()
simclr_model = simclr_model.cuda()

# Remove final projection head → we want the 2048-dim representation
feature_extractor = nn.Sequential(*list(simclr_model.children())[:-1])

# Dataset + Loader
dataset_simclr = SimCLRFingerprintDataset(file_list, transform=get_simclr_augs())
loader_simclr = DataLoader(dataset_simclr, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True, drop_last=False)

# Extract features
all_feats = []
with torch.no_grad():
    for view1, _ in tqdm(loader_simclr, desc="SimCLR → 6000 images"):
        x = view1.repeat(1, 3, 1, 1).cuda()          # (B,1,224,224) → (B,3,224,224)
        feats = feature_extractor(x)
        feats = feats.squeeze(-1).squeeze(-1)        # (B,2048)
        all_feats.append(feats.cpu().numpy())

simclr_features_full = np.concatenate(all_feats, axis=0)
print(f"Success! SimCLR features shape: {simclr_features_full.shape}")  # → (6000, 2048)

# Save
np.save("simclr_features_full_6000.npy", simclr_features_full)
print("Saved as simclr_features_full_6000.npy")